# New Implementation of DataFrameEncoder


This time, this class will not be responsible for any splitting purposes.
Instead, it will purely act as a column-wise / group-wise transformation.

In [1]:
%config InteractiveShell.ast_node_interactivity='last_expr_or_assign'  # always print last expr.
%config InlineBackend.figure_format = 'svg'
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [2]:
import logging
logging.basicConfig(level=logging.INFO)

In [3]:
import logging
from abc import ABC, abstractmethod
from collections import defaultdict, namedtuple
from collections.abc import Callable, Collection, Hashable, Iterable, Mapping, Sequence
from functools import singledispatchmethod
from typing import Any, Final, Generic, Literal, Optional, Union, overload

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pandas.api.types
from pandas.core.indexes.frozen import FrozenList

import torch
from pandas import (
    NA,
    DataFrame,
    DatetimeIndex,
    Index,
    MultiIndex,
    Series,
    Timedelta,
    Timestamp,
)
from torch import Tensor

from tsdm.datasets import TimeTensor
from tsdm.encoders.modular import *
from tsdm.encoders.modular import BaseEncoder
from tsdm.util.types import PathType
from tsdm.util.types.abc import HashableType

np.set_printoptions(precision=4, floatmode="fixed", suppress=True)
rng = np.random.default_rng()

INFO:numexpr.utils:Note: NumExpr detected 24 cores but "NUMEXPR_MAX_THREADS" not set, so enforcing safe limit of 8.
INFO:numexpr.utils:NumExpr defaulting to 8 threads.
INFO:tsdm.config._config:Available Models: {'SET-TS', 'IP-Net', 'NCDE', 'TFT', 'N-BEATS', 'GRU-ODE-Bayes', 'M-RNN', 'Informer', 'Latent-ODE', 'mTAN', 'ODE-RNN', 'BRITS', 'NODE', 'TPA'}
INFO:tsdm.config._config:Available Datasets: {'Character Trajectories', 'M5', 'Physionet 2012', 'MuJoCo', 'Traffic', 'USHCN', 'Air Quality Multi-Site', 'Household Consumption', 'tourism2', 'tourism1', 'UWAVE', 'M3', 'Physionet 2019', 'Electricity', 'M4', 'Human Activity', 'Air Quality'}
INFO:tsdm.config._config:Initializing folder structure
INFO:tsdm.config._config:creating folder /home/rscholz/.tsdm/datasets
INFO:tsdm.config._config:creating folder /home/rscholz/.tsdm/models
INFO:tsdm.config._config:creating folder /home/rscholz/.tsdm/logs
INFO:tsdm.config._config:creating folder /home/rscholz/.tsdm/rawdata
INFO:tsdm.config._config:Create

Generator(PCG64) at 0x7F2A8114C120

In [49]:
class FrameEncoder(BaseEncoder):
    KEYS = Hashable
    columns: Index
    dtypes: Series
    index_columns: Index
    index_dtypes: Series

    column_encoders: Optional[Mapping[tuple[KEYS, ...], BaseEncoder]]
    r"""Encoders for the columns."""
    index_encoders: Optional[Mapping[tuple[KEYS, ...], BaseEncoder]]
    r"""Optional Encoder for the index."""
    column_decoders: Optional[Mapping[tuple[KEYS, ...], BaseEncoder]]
    r"""Reverse Dictionary from encoded column name -> encoder"""
    index_decoders: Optional[Mapping[tuple[KEYS, ...], BaseEncoder]]
    r"""Reverse Dictionary from encoded index name -> encoder"""

    @staticmethod
    def _names(obj) -> Union[str, Iterable[str]]:
        if isinstance(obj, MultiIndex):
            return FrozenList(obj.names)
        if isinstance(obj, (Series, Index)):
            return obj.name
        if isinstance(obj, DataFrame):
            return FrozenList(obj.columns)
        raise ValueError
    
    def __init__(
        self,
        column_encoders: Optional[
            Union[BaseEncoder, Mapping[Union[KEYS, Collection[KEYS]], BaseEncoder]]
        ] = None,
        *,
        index_encoders: Optional[
            Union[BaseEncoder, Mapping[Union[KEYS, Collection[KEYS]], BaseEncoder]]
        ] = None,
    ):
        super().__init__()
        self.column_encoders = column_encoders
        self.index_encoders = index_encoders

    def fit(self, data: DataFrame) -> None:
        data = data.copy()
        index = data.index.to_frame()
        self.columns = data.columns
        self.dtypes = data.dtypes
        self.index_columns = index.columns
        self.index_dtypes = index.dtypes

        if self.column_encoders is None:
            self.column_decoders = None
        elif isinstance(self.column_encoders, BaseEncoder):
            self.column_encoders.fit(data)
            self.column_decoders = self.column_encoders
        else:
            self.column_decoders = {}
            for group, encoder in self.column_encoders.items():
                encoder.fit(data[group])
                encoded = encoder.encode(data[group])
                self.column_decoders[self._names(encoded)] = encoder

        if self.index_encoders is None:
            self.index_decoders = None
        elif isinstance(self.index_encoders, BaseEncoder):
            self.index_encoders.fit(index)
            self.index_decoders = self.index_encoders
        else:
            self.index_decoders = {}
            for group, encoder in self.index_encoders.items():
                encoder.fit(index[group])
                encoded = encoder.encode(index[group])
                self.index_decoders[self._names(encoded)] = encoder
         
    def encode(self, data: DataFrame) -> DataFrame:
        data = data.copy(deep=True)
        index = data.index.to_frame()
        encoded_cols = data
        encoded_inds = encoded_cols.index.to_frame()

        if self.column_encoders is None:
            pass
        elif isinstance(self.column_encoders, BaseEncoder):
            encoded = self.column_encoders.encode(data)
            encoded_cols = encoded_cols.drop(columns=data.columns)
            encoded_cols[self._names(encoded)] = encoded
        else:
            for group, encoder in self.column_encoders.items():
                encoded = encoder.encode(data[group])
                encoded_cols = encoded_cols.drop(columns=group)
                encoded_cols[self._names(encoded)] = encoded

        if self.index_encoders is None:
            pass
        elif isinstance(self.index_encoders, BaseEncoder):
            encoded = self.index_encoders.encode(index)
            encoded_inds = encoded_inds.drop(columns=index.columns)
            encoded_inds[self._names(encoded)] = encoded
        else:
            for group, encoder in self.index_encoders.items():
                encoded = encoder.encode(index[group])
                encoded_inds = encoded_inds.drop(columns=group)
                encoded_inds[self._names(encoded)] = encoded

        # Assemble DataFrame
        encoded = DataFrame(encoded_cols)
        encoded[self._names(encoded_inds)] = encoded_inds
        encoded = encoded.set_index(self._names(encoded_inds))
        return encoded

    def decode(self, data: DataFrame) -> DataFrame:
        data = data.copy(deep=True)
        index = data.index.to_frame()
        decoded_cols = data
        decoded_inds = decoded_cols.index.to_frame()
        
        if self.column_decoders is None:
            pass
        elif isinstance(self.column_decoders, BaseEncoder):
            decoded = self.column_decoders.decode(data)
            decoded_cols = decoded_cols.drop(columns=data.columns)
            decoded_cols[self._names(decoded)] = decoded
        else:
            for group, encoder in self.column_decoders.items():
                decoded = encoder.decode(data[group])
                decoded_cols = decoded_cols.drop(columns=group)
                decoded_cols[self._names(decoded)] = decoded

        if self.index_decoders is None:
            pass
        elif isinstance(self.index_decoders, BaseEncoder):
            decoded = self.index_decoders.decode(index)
            decoded_inds = decoded_inds.drop(columns=index.columns)
            decoded_inds[self._names(decoded)] = decoded
        else:
            for group, encoder in self.index_decoders.items():
                decoded = encoder.decode(index[group])
                decoded_inds = decoded_inds.drop(columns=group)
                decoded_inds[self._names(decoded)] = decoded
        
        # Restore index order + dtypes
        decoded_inds = decoded_inds[self.index_columns]
        decoded_inds = decoded_inds.astype(self.index_dtypes)
        
        # Assemble DataFrame
        decoded = DataFrame(decoded_cols)
        decoded[self._names(decoded_inds)] = decoded_inds
        decoded = decoded.set_index(self._names(decoded_inds))
        decoded = decoded[self.columns]
        decoded = decoded.astype(self.dtypes)

        return decoded

In [50]:
from tsdm.tasks import KIWI_FINAL_PRODUCT

task = KIWI_FINAL_PRODUCT()
ts = task.timeseries.sort_index(axis="index").sort_index(axis="columns")
channel_freq = pd.notna(ts).mean().sort_values()
fast_channels = FrozenList(channel_freq[channel_freq >= 0.1].index)
slow_channels = FrozenList(channel_freq[channel_freq < 0.1].index)
FAST = ts[fast_channels].dropna(how="all")
SLOW = ts[slow_channels].dropna(how="all")
groups = {"fast": fast_channels, "slow": slow_channels}

INFO:tsdm.datasets.base._base:KIWI_RUNS: START cleaning dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/timeseries already exists, skipping.
INFO:tsdm.datasets.base._base:KIWI_RUNS/metadata already exists, skipping.
INFO:tsdm.datasets.base._base:KIWI_RUNS/units already exists, skipping.
INFO:tsdm.datasets.base._base:KIWI_RUNS: DONE cleaning dataset
INFO:tsdm.datasets.base._base:KIWI_RUNS: START loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/timeseries: START loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/timeseries: DONE loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/metadata: START loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/metadata: DONE loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/units: START loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS/units: DONE loading dataset!
INFO:tsdm.datasets.base._base:KIWI_RUNS: DONE loading dataset!
INFO:tsdm.tasks.kiwi_final_product:(510, 16849): no t_induction!
INFO:tsdm.tasks.kiwi_fi

{'fast': FrozenList(['Temperature', 'DOT', 'pH', 'Cumulated_feed_volume_glucose', 'Cumulated_feed_volume_medium', 'Flow_Air', 'InducerConcentration', 'Probe_Volume', 'StirringSpeed']),
 'slow': FrozenList(['OD600', 'Fluo_GFP', 'Glucose', 'Acetate', 'Base', 'Volume'])}

In [62]:
from tsdm.encoders.modular import *
enc = FrameEncoder(
    column_encoders = {
        fast_channels: Standardizer(),
        slow_channels: MinMaxScaler(),
    },
    index_encoders = {
        "run_id": IntEncoder(),
        "experiment_id" : IntEncoder(),
        "measurement_time" : DateTimeEncoder(),
    }
)
enc.fit(ts)

[autoreload of tsdm.encoders.modular failed: Traceback (most recent call last):
  File "/home/rscholz/miniconda3/envs/kiwi/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 257, in check
    superreload(m, reload, self.old_objects)
  File "/home/rscholz/miniconda3/envs/kiwi/lib/python3.9/site-packages/IPython/extensions/autoreload.py", line 455, in superreload
    module = reload(module)
  File "/home/rscholz/miniconda3/envs/kiwi/lib/python3.9/importlib/__init__.py", line 169, in reload
    _bootstrap._exec(spec, module)
  File "<frozen importlib._bootstrap>", line 613, in _exec
  File "<frozen importlib._bootstrap_external>", line 850, in exec_module
  File "<frozen importlib._bootstrap>", line 228, in _call_with_frames_removed
  File "/home/rscholz/Projects/KIWI/tsdm/tsdm/encoders/modular/__init__.py", line 40, in <module>
    from tsdm.encoders.modular._modular import (
ImportError: cannot import name 'FrameSplitter' from 'tsdm.encoders.modular._modular' (/home/rsc

In [63]:
encoded = enc.encode(ts)
decoded = enc.decode(encoded)
pd.testing.assert_frame_equal(ts, decoded)

In [55]:
encoded

variable                               Temperature       DOT        pH  \
run_id experiment_id measurement_time                                    
439    15325         0.0                 -1.924218       NaN       NaN   
                     10.0                      NaN -2.588519  1.580949   
                     15.0                      NaN       NaN       NaN   
                     16.0                -2.424592       NaN       NaN   
                     25.0                      NaN -2.588519  1.641594   
...                                            ...       ...       ...   
510    16871         27783077.0                NaN       NaN       NaN   
                     27783122.0           0.115787       NaN       NaN   
                     27783131.0                NaN  0.216809  1.885555   
                     27783193.0           0.096537       NaN       NaN   
                     27783202.0                NaN  0.195626  1.983697   

variable                               Cumulated_feed_volume_glucose  \
run_id experiment_id measurement_time                                  
439    15325         0.0                                   -0.615802   
                     10.0                                  -0.615802   
                     15.0                                  -0.615802   
                     16.0                                  -0.615802   
                     25.0                                  -0.615802   
...                                                              ...   
510    16871         27783077.0                             1.393977   
                     27783122.0                             1.393977   
                     27783131.0                             1.393977   
                     27783193.0                             1.393977   
                     27783202.0                             1.393977   

variable                               Cumulated_feed_volume_medium  Flow_Air  \
run_id experiment_id measurement_time                                           
439    15325         0.0                                  -1.272827 -2.848577   
                     10.0                                 -1.272827 -2.848577   
                     15.0                                 -1.272827 -2.848577   
                     16.0                                 -1.272827 -2.848577   
                     25.0                                 -1.272827 -2.848577   
...                                                             ...       ...   
510    16871         27783077.0                            2.033456 -0.422224   
                     27783122.0                            2.033456 -0.422224   
                     27783131.0                            2.033456 -0.422224   
                     27783193.0                            2.033456 -0.422224   
                     27783202.0                            2.033456 -0.422224   

variable                               InducerConcentration  Probe_Volume  \
run_id experiment_id measurement_time                                       
439    15325         0.0                          -0.450452     -1.473321   
                     10.0                         -0.450452     -1.473321   
                     15.0                         -0.450452     -1.473321   
                     16.0                         -0.450452     -1.473321   
                     25.0                         -0.450452     -1.473321   
...                                                     ...           ...   
510    16871         27783077.0                   -0.450452      2.626016   
                     27783122.0                   -0.450452      2.626016   
                     27783131.0                   -0.450452      2.626016   
                     27783193.0                   -0.450452      2.626016   
                     27783202.0                   -0.450452      2.626016   

variable                               Stirr

In [ ]:
T = ts[[]].reset_index(-1)["measurement_time"]

In [ ]:
e = DateTimeEncoder()
e.fit(T)
e.decode(e.encode(T))

In [43]:
mask = pd.notna(ts.Acetate)
ts.Acetate[mask]

run_id  experiment_id  measurement_time   
439     15325          2020-12-09 09:48:38    0.115714
                       2020-12-09 11:18:43    0.157560
                       2020-12-09 12:48:48    0.109903
                       2020-12-09 14:18:48    0.063760
                       2020-12-09 15:49:34    0.046601
                                                ...   
510     16871          2021-10-26 19:00:55    0.079195
                       2021-10-26 20:06:00    0.175493
                       2021-10-26 20:17:10    0.075955
                       2021-10-26 21:30:40    0.061150
                       2021-10-26 22:30:32    0.049915
Name: Acetate, Length: 1676, dtype: float64

In [58]:
from types import MethodType
ts.to_frame = MethodType(lambda self: self, ts)

In [63]:
%%timeit
ts.to_frame()

67.1 ns ± 0.334 ns per loop (mean ± std. dev. of 7 runs, 10,000,000 loops each)


In [9]:
e = MinMaxScaler()

e.fit(ts[fast_channels])

In [10]:
e.xmin.shape

(9,)

In [11]:
e.ymin.ndim

0

MinMaxScaler([-1])

In [20]:
e[3:6].xmin

array([0.0000, 0.0000, 0.0000])

In [11]:
e.encode(ts[fast_channels])

variable                                  Temperature     DOT        pH  \
run_id experiment_id measurement_time                                     
439    15325         2020-12-09 09:10:09     0.425777     NaN       NaN   
                     2020-12-09 09:10:19          NaN  0.0000  0.534965   
                     2020-12-09 09:10:24          NaN     NaN       NaN   
                     2020-12-09 09:10:25     0.395857     NaN       NaN   
                     2020-12-09 09:10:34          NaN  0.0000  0.538462   
...                                               ...     ...       ...   
510    16871         2021-10-26 22:41:26          NaN     NaN       NaN   
                     2021-10-26 22:42:11     0.547756     NaN       NaN   
                     2021-10-26 22:42:20          NaN  0.7946  0.552528   
                     2021-10-26 22:43:22     0.546605     NaN       NaN   
                     2021-10-26 22:43:31          NaN  0.7886  0.558186   

variable                                  Cumulated_feed_volume_glucose  \
run_id experiment_id measurement_time                                     
439    15325         2020-12-09 09:10:09                       0.000000   
                     2020-12-09 09:10:19                       0.000000   
                     2020-12-09 09:10:24                       0.000000   
                     2020-12-09 09:10:25                       0.000000   
                     2020-12-09 09:10:34                       0.000000   
...                                                                 ...   
510    16871         2021-10-26 22:41:26                       0.258717   
                     2021-10-26 22:42:11                       0.258717   
                     2021-10-26 22:42:20                       0.258717   
                     2021-10-26 22:43:22                       0.258717   
                     2021-10-26 22:43:31                       0.258717   

variable                                  Cumulated_feed_volume_medium  \
run_id experiment_id measurement_time                                    
439    15325         2020-12-09 09:10:09                      0.000000   
                     2020-12-09 09:10:19                      0.000000   
                     2020-12-09 09:10:24                      0.000000   
                     2020-12-09 09:10:25                      0.000000   
                     2020-12-09 09:10:34                      0.000000   
...                                                                ...   
510    16871         2021-10-26 22:41:26                      0.748354   
                     2021-10-26 22:42:11                      0.748354   
                     2021-10-26 22:42:20                      0.748354   
                     2021-10-26 22:43:22                      0.748354   
                     2021-10-26 22:43:31                      0.748354   

variable                                  Flow_Air  InducerConcentration  \
run_id experiment_id measurement_time                                      
439    15325         2020-12-09 09:10:09       0.0                   0.0   
                     2020-12-09 09:10:19       0.0                   0.0   
                     2020-12-09 09:10:24       0.0                   0.0   
                     2020-12-09 09:10:25       0.0                   0.0   
                     2020-12-09 09:10:34       0.0                   0.0   
...                                            ...                   ...   
510    16871         2021-10-26 22:41:26       0.5                   0.0   
                     2021-10-26 22:42:11       0.5                   0.0   
                     2021-10-26 22:42:20       0.5                   0.0   
                     2021-10-26 22:43:22       0.5                   0.0   
                     2021-10-26 22:43:31       0.5                   0.0   

variable                                  Probe_Volume  StirringSpeed  
run_id experiment_id mea